In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
jdbc_url = (
    "jdbc:sqlserver://complaints-sql-server.database.windows.net:1433;"
    "databaseName=customer-complaints-db;"
    "encrypt=true;trustServerCertificate=false;loginTimeout=30"
)

jdbc_props = {
    "user":     dbutils.secrets.get(scope="complaints_jdbc", key="sql_user"),
    "password": dbutils.secrets.get(scope="complaints_jdbc", key="sql_password"),
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
from pyspark.sql.functions import current_timestamp, lit, max as spark_max

checkpoint_table = "bronze.complaints_checkpoint"
target_table     = "bronze.customer_complaints"

if spark.catalog.tableExists(checkpoint_table):
    last_date = spark.table(checkpoint_table).collect()[0]["last_date"]
    print(f"Incremental load — closed days after: {last_date}")
else:
    last_date = "1900-01-01"
    print("First run — full load")

In [0]:
import urllib.request
print(urllib.request.urlopen("https://api.ipify.org").read().decode())

In [0]:
query = f"""
(
    SELECT * FROM dbo.customer_complaints
    WHERE Date_received >= '{last_date}'
) AS incremental_batch
"""

df_bronze = spark.read.jdbc(url=jdbc_url, table=query, properties=jdbc_props)
row_count = df_bronze.count()
print(f"Rows pulled (includes boundary re-check): {row_count}")

In [0]:
df_bronze = (df_bronze
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("azure_sql")))

df_bronze.createOrReplaceTempView("incremental_batch")


In [0]:
target = "bronze.customer_complaints"

if spark.catalog.tableExists(target):
    result = spark.sql(f"""
        MERGE INTO {target} AS t
        USING incremental_batch AS s
        ON t.Complaint_ID = s.Complaint_ID
        WHEN NOT MATCHED THEN INSERT *
    """)
    inserted = result.collect()[0]["num_inserted_rows"]
else:
    # very first run — no table yet, so just land it
    df_bronze.write.format("delta").mode("overwrite").saveAsTable(target)
    inserted = df_bronze.count()

print(f"New rows inserted: {inserted}  |  duplicates skipped: {row_count - inserted}")

if inserted == 0:
    print("No new complaints. Pipeline complete.")
    dbutils.notebook.exit("No new data")

In [0]:
new_max = df_bronze.agg(spark_max("Date_received")).collect()[0][0]

spark.createDataFrame([(str(new_max),)], ["last_date"]) \
     .write.format("delta").mode("overwrite") \
     .saveAsTable(checkpoint_table)

print(f"Checkpoint updated to: {new_max}")

In [0]:
%sql
select * from bronze.customer_complaints limit 10